
# KV Cache Speed Benchmark on a Long Contract

**Day 1 — AI Foundations · Practical 4 of 6 · Companion to the "Finetuning & KV Cache" deck**

> **Running in Google Colab:** Sections 1-6 work fine on the default **CPU runtime**. Section 7
> (Flash Attention) only produces a meaningful result on a **GPU runtime** (Runtime → Change
> runtime type → GPU) — it's guarded and will just print a skip message otherwise.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Measure, with your own eyes, the real speed difference between generation **with** and
   **without** KV caching
2. Explain why the gap gets **worse as the document gets longer** — directly relevant to legal
   documents, which are rarely short
3. Plot latency vs. sequence length for both approaches
4. (Optional, GPU-only) Compare attention backends to see Flash Attention's real-world effect

## Why This Matters for a Law Firm

Legal documents are long. A contract-drafting or contract-summarization assistant that
regenerates from scratch at every single word will get progressively, painfully slower as the
document grows — exactly when it matters most (a 40-page master service agreement, not a
one-line email). KV caching is the difference between an assistant that feels instant and one
that feels unusable on real documents.

## Notebook Workflow

```mermaid
flowchart TD
    A["Long contract-style prompt\n(simulated section text)"] --> B["Generate WITH\nuse_cache=True"]
    A --> C["Generate WITHOUT\nuse_cache=False"]
    B --> D["Record latency"]
    C --> D
    D --> E["Repeat at increasing\nprompt lengths"]
    E --> F["Plot: latency vs.\nsequence length"]
    F --> G["Takeaway: gap widens\nas documents get longer"]



## Section 1 — Setup

We reuse **DistilGPT2** for this benchmark — its small size means we can run meaningful timing
comparisons quickly, even on a CPU-only laptop, while the *relative* speedup pattern from
KV caching holds at any model scale.


In [ ]:

# Install dependencies.
# Running in Google Colab: this cell installs everything needed -- just run it.
%pip install -q transformers torch matplotlib

import time
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import matplotlib.pyplot as plt

torch.manual_seed(42)

MODEL_NAME = "distilgpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.eval()

print(f"Loaded {MODEL_NAME}. Running on: {'GPU' if torch.cuda.is_available() else 'CPU'}")



## Section 2 — A Long, Realistic Legal Prompt

The paragraph below simulates the opening of a real contract section — the kind of
multi-clause block a drafting assistant would realistically need to read as context before
continuing to generate. Its length (not its exact content) is what matters for this benchmark.


In [ ]:

long_contract_context = (
    "This Master Services Agreement (the \"Agreement\") is entered into as of the Effective "
    "Date by and between the Client and the Contractor. WHEREAS, the Client desires to "
    "engage the Contractor to provide certain professional services as more particularly "
    "described in one or more Statements of Work to be executed by the parties from time to "
    "time hereunder; and WHEREAS, the Contractor is willing to provide such services subject "
    "to the terms and conditions set forth herein; NOW, THEREFORE, in consideration of the "
    "mutual covenants and agreements contained herein, and for other good and valuable "
    "consideration, the receipt and sufficiency of which are hereby acknowledged, the "
    "parties agree as follows: 1. Services. The Contractor shall perform the services "
    "described in each Statement of Work in a professional and workmanlike manner consistent "
    "with generally accepted industry standards. 2. Term. This Agreement shall commence on "
    "the Effective Date and shall continue in full force and effect until terminated in "
    "accordance with Section 8 hereof. 3. Compensation. In consideration for the services "
    "performed hereunder, the Client shall pay the Contractor the fees set forth in the "
    "applicable Statement of Work, subject to the payment terms specified therein."
)

print(f"Context length in characters: {len(long_contract_context)}")
print(f"Context length in tokens:     {len(tokenizer.encode(long_contract_context))}")



## Section 3 — Timing Helper

We write a small helper that generates a fixed number of new tokens given a prompt, with
`use_cache` toggled on or off, and returns the wall-clock time taken. `torch.no_grad()` is used
throughout since we're only running inference, never backpropagation.


In [ ]:

def timed_generate(model, prompt, max_new_tokens, use_cache):
    inputs = tokenizer(prompt, return_tensors="pt")

    start = time.perf_counter()
    with torch.no_grad():
        model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=use_cache,
            do_sample=False,          # greedy decoding: deterministic, removes sampling noise from timing
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start
    return elapsed



## Section 4 — Single Head-to-Head Comparison

Let's first compare the two modes at a single prompt length, generating a fixed number of new
tokens, so we can see the raw magnitude of the difference before looking at the trend.


In [ ]:

MAX_NEW_TOKENS = 60

time_with_cache = timed_generate(model, long_contract_context, MAX_NEW_TOKENS, use_cache=True)
time_without_cache = timed_generate(model, long_contract_context, MAX_NEW_TOKENS, use_cache=False)

speedup = time_without_cache / time_with_cache

print(f"Generating {MAX_NEW_TOKENS} new tokens after a {len(tokenizer.encode(long_contract_context))}-token prompt:\n")
print(f"  WITH KV cache:    {time_with_cache:.3f} sec")
print(f"  WITHOUT KV cache: {time_without_cache:.3f} sec")
print(f"  Speedup:          {speedup:.2f}x")



## Section 5 — How Does the Gap Change as the Document Gets Longer?

Now the key experiment: repeat the same comparison, but progressively grow the **prompt
length** by truncating/extending our contract context, and hold the number of newly generated
tokens fixed. The deck's claim is that the no-cache approach scales worse (roughly O(n²)) as
context length grows — let's see if that holds up empirically on our own machine.


In [ ]:

# Build a few progressively longer prompts by repeating the base context text
prompt_lengths_tokens = []
cache_times = []
no_cache_times = []

repeat_counts = [1, 2, 3, 4]  # how many times to repeat the base paragraph
MAX_NEW_TOKENS_SWEEP = 30      # keep new-token count fixed across the sweep

for reps in repeat_counts:
    prompt_text = (long_contract_context + " ") * reps
    prompt_token_len = len(tokenizer.encode(prompt_text))

    t_cache = timed_generate(model, prompt_text, MAX_NEW_TOKENS_SWEEP, use_cache=True)
    t_no_cache = timed_generate(model, prompt_text, MAX_NEW_TOKENS_SWEEP, use_cache=False)

    prompt_lengths_tokens.append(prompt_token_len)
    cache_times.append(t_cache)
    no_cache_times.append(t_no_cache)

    print(f"Prompt length: {prompt_token_len:>5} tokens   "
          f"| with cache: {t_cache:.3f}s   | without cache: {t_no_cache:.3f}s   "
          f"| speedup: {t_no_cache/t_cache:.2f}x")



## Section 6 — Plot the Trend

A plot makes the growing gap much easier to see than a table of numbers.


In [ ]:

plt.figure(figsize=(9, 6))
plt.plot(prompt_lengths_tokens, cache_times, marker="o", label="WITH KV cache", linewidth=2)
plt.plot(prompt_lengths_tokens, no_cache_times, marker="o", label="WITHOUT KV cache", linewidth=2)

plt.xlabel("Prompt (document context) length — tokens")
plt.ylabel(f"Time to generate {MAX_NEW_TOKENS_SWEEP} new tokens (seconds)")
plt.title("KV Cache Impact Grows With Document Length")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: the 'WITHOUT KV cache' line climbs faster than the 'WITH KV cache' line as")
print("prompt length increases -- exactly the O(n^2) vs. near-linear behavior from the deck.")



## Section 7 (Optional) — Flash Attention Backend Comparison

This section only produces a meaningful difference **on a CUDA GPU** — Flash Attention's
speedup comes from GPU memory-bandwidth optimization (HBM vs. SRAM), which doesn't apply the
same way on CPU. If you're running on a laptop CPU, skip this section or run it purely to see
that the backend selection code executes without error.

We use PyTorch's built-in `torch.nn.functional.scaled_dot_product_attention`, which can
dispatch to a Flash-Attention-style kernel automatically when the hardware supports it.


In [ ]:

if torch.cuda.is_available():
    # torch.nn.attention.sdpa_kernel is only available on newer PyTorch (>= 2.3).
    # Colab's preinstalled torch version varies by image, so we fail gracefully
    # rather than crash the whole notebook run if the API isn't present.
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        sdpa_api_available = True
    except ImportError:
        sdpa_api_available = False
        print(f"torch {torch.__version__} does not expose torch.nn.attention.sdpa_kernel.")
        print("Upgrade PyTorch (%pip install -q -U torch) to run this comparison, or skip it --")
        print("the KV cache results in Sections 4-6 already cover the core concept.")

    if sdpa_api_available:
        model_gpu = model.to("cuda")
        inputs_gpu = tokenizer(long_contract_context, return_tensors="pt").to("cuda")

        def timed_sdpa_generate(backend):
            with sdpa_kernel(backend):
                start = time.perf_counter()
                with torch.no_grad():
                    model_gpu.generate(
                        **inputs_gpu,
                        max_new_tokens=MAX_NEW_TOKENS,
                        use_cache=True,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                return time.perf_counter() - start

        flash_time = timed_sdpa_generate(SDPBackend.FLASH_ATTENTION)
        math_time = timed_sdpa_generate(SDPBackend.MATH)

        print(f"Flash Attention backend: {flash_time:.3f} sec")
        print(f"Standard 'math' backend: {math_time:.3f} sec")
        print(f"Speedup: {math_time / flash_time:.2f}x")
else:
    print("No CUDA GPU detected -- skipping Flash Attention backend comparison.")
    print("In Colab: Runtime -> Change runtime type -> GPU, then re-run this cell.")
    print("This section is optional and illustrative only; the KV cache results above")
    print("(Sections 4-6) already demonstrate the core inference-optimization concept.")



## Key Takeaways

1. **KV caching provides a real, measurable speedup** — and you just measured it yourself,
   not just read about it in the deck.
2. **The benefit compounds with document length.** For a law firm working with long contracts,
   opinion letters, and briefs, this isn't a minor optimization — it's the difference between
   an interactive tool and an unusable one on real-length documents.
3. **Flash Attention** attacks a different bottleneck (GPU memory bandwidth) and compounds on
   top of KV caching — both are usually enabled by default in production inference engines
   (vLLM, PyTorch's `scaled_dot_product_attention`), so in practice you get both for free.
4. When evaluating any LLM deployment (self-hosted or vendor API) for long-document legal
   workflows, ask directly whether KV caching and Flash-Attention-class optimizations are
   enabled — the difference shows up immediately at document lengths lawyers actually use.

**Next up:** the *Modern Model Landscape* notebook — a quick hands-on tour of a legal-domain
classification model and a small on-device model.
